# Simple GAN — Unconditional Face Generation on CelebA
**Architecture:** DCGAN (Radford et al., 2015)
**Dataset:** CelebA (torchvision) · **Framework:** PyTorch · **Platform:** Google Colab

---
## Purpose
This is the **baseline GAN** before AttGAN. Comparing both shows the advantage of
conditional attribute-guided generation.

| | Simple GAN (this notebook) | AttGAN |
|---|---|---|
| Conditioning | None — random noise only | Attribute vector {-1, +1} |
| Control over output | None | Edit 13 specific facial attributes |
| Loss | BCE adversarial only | Adv + Classification + Reconstruction |
| Output resolution | 64x64 | 128x128 |

---
## Before you start
1. **Set runtime to GPU** → `Runtime → Change runtime type → T4 GPU → Save`
2. Run all cells top to bottom
3. CelebA downloads automatically in Cell 3 (~1.4 GB, first run only)

> If Cell 3 raises `FileURLRetrievalError`, run the **fallback cell** at the bottom.

---
## Cell 1 — Clone repo & install

In [ ]:
!git clone https://github.com/YOUR_USERNAME/GAN_Project_DL.git
%cd GAN_Project_DL
!pip install -q -r requirements.txt

---
## Cell 2 — Verify GPU & set config

In [ ]:
import torch, sys
print(f'Python  : {sys.version.split()[0]}')
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    raise RuntimeError('No GPU found — go to Runtime > Change Runtime Type > T4 GPU')

from config import Config

cfg = Config()
cfg.EXPERIMENT_NAME = 'simple_gan'
cfg.__init__()   # creates results/simple_gan/ and checkpoints/simple_gan/

device = torch.device('cuda')

# Optional: reduce for a quick smoke test
# cfg.N_EPOCHS = 5

print(f'Experiment  : {cfg.EXPERIMENT_NAME}')
print(f'Epochs      : {cfg.N_EPOCHS}')
print(f'Batch size  : {cfg.BATCH_SIZE}')
print(f'Results dir : {cfg.RESULTS_DIR}')

---
## Cell 3 — Download & load CelebA
torchvision downloads CelebA automatically on first run (~1.4 GB).
Images are loaded at 128x128 and downsampled to 64x64 inside the training loop
(the DCGAN architecture expects 64x64 input).

> If you get `FileURLRetrievalError`, run the **fallback cell** at the bottom of this notebook.

In [ ]:
from src.dataset import get_loaders

train_loader, test_loader = get_loaders(cfg)

imgs, attrs = next(iter(train_loader))
print(f'Image batch : {imgs.shape}   (loaded at 128x128, downsampled to 64x64 during training)')
print(f'Pixel range : [{imgs.min():.2f}, {imgs.max():.2f}]')

---
## Cell 4 — Build models
```
noise z (100 x 1 x 1)
  |--> Generator  (5x ConvTranspose2d + BatchNorm + ReLU -> Tanh)
       fake image (3 x 64 x 64)
         |--> Discriminator  (5x Conv2d + BatchNorm + LeakyReLU -> Sigmoid)
              real/fake scalar [0, 1]

Loss G : BCE(D(G(z)), 1)
Loss D : BCE(D(x), 1) + BCE(D(G(z)), 0)
```

In [ ]:
from src.simple_gan import build_simple_models

LATENT_DIM = 100
gen, dis = build_simple_models(latent_dim=LATENT_DIM, device=device)

---
## Cell 5 — Train
The SimpleGAN uses standard binary cross-entropy adversarial loss.
Sample grids are saved to `results/simple_gan/` every 5 epochs.

Tip: a healthy training run shows G loss and D loss both decreasing and
eventually oscillating around a stable value — neither should go to zero.

In [ ]:
from src.simple_gan import train_simple_gan

g_losses, d_losses = train_simple_gan(
    gen, dis, train_loader, cfg, device, latent_dim=LATENT_DIM
)

---
## Cell 6 — Loss curves

In [ ]:
from src.utils import plot_losses
plot_losses(g_losses, d_losses, cfg)

---
## Cell 7 — FID & DACID metrics
**FID** (Frechet Inception Distance): standard GAN quality metric. Lower = better.

**DACID** (Dany Aissa & Clara's Image Distance): custom metric — L2 distance between
mean Inception feature vectors. Faster than FID. Lower = better.

> Takes ~5 extra minutes. Set `cfg.COMPUTE_METRICS = False` to skip.

In [ ]:
# cfg.COMPUTE_METRICS = False  # uncomment to skip

if cfg.COMPUTE_METRICS:
    from src.metrics import compute_metrics_simple_gan
    scores = compute_metrics_simple_gan(gen, test_loader, LATENT_DIM, cfg, device)
    print(f"FID   : {scores['fid']}   (lower = better)")
    print(f"DACID : {scores['dacid']}   (lower = better)")
else:
    scores = {}
    print('Metrics skipped.')

---
## Cell 8 — Save checkpoint & metrics.json

In [ ]:
import json, torch

# Save model weights
ckpt = cfg.CHECKPOINT_DIR / 'simple_gan_final.pt'
torch.save({'gen': gen.state_dict(), 'dis': dis.state_dict()}, ckpt)
print(f'Checkpoint -> {ckpt}')

# Save metrics for export_results.py
payload = {
    'experiment': cfg.EXPERIMENT_NAME,
    'model':      'SimpleGAN',
    'fid':        scores.get('fid'),
    'dacid':      scores.get('dacid'),
    'g_losses':   g_losses,
    'd_losses':   d_losses,
}
out = cfg.RESULTS_DIR / 'metrics.json'
with open(out, 'w') as f:
    json.dump(payload, f, indent=2)
print(f'Metrics  -> {out}')

---
## Cell 9 — Download results to your computer

In [ ]:
import shutil
from google.colab import files

zip_name = 'simple_gan_results'
shutil.make_archive(zip_name, 'zip', cfg.RESULTS_DIR)
files.download(f'{zip_name}.zip')
print(f'Downloaded {zip_name}.zip')

---
## Fallback — CelebA download quota exceeded
If Cell 3 raises `FileURLRetrievalError: Too many users have viewed or downloaded this file`,
use one of these alternatives:

In [ ]:
# Option A — Kaggle API
# 1. Get your Kaggle token: kaggle.com -> Account -> Create New Token -> downloads kaggle.json
# 2. Upload kaggle.json to Colab (Files panel, left sidebar), then run:

# !mkdir -p ~/.kaggle
# !cp kaggle.json ~/.kaggle/
# !chmod 600 ~/.kaggle/kaggle.json
# !pip install -q kaggle
# !kaggle datasets download -d jessicali9530/celeba-dataset -p data/
# !unzip -q data/celeba-dataset.zip -d data/

print('Uncomment lines above after uploading kaggle.json')

In [ ]:
# Option B — Google Drive (if you have CelebA already saved there)

# from google.colab import drive
# drive.mount('/content/drive')
# import os
# os.makedirs('data', exist_ok=True)
# !ln -sfn /content/drive/MyDrive/celeba data/celeba
# Then re-run Cell 3

print('Uncomment lines above after mounting Drive')